# 02 — Shared Train/Val Split (seed 42)

The test split is unlabeled, so **all model comparisons (03/04/05) are run on this single, deterministically generated train/val split** from the labeled train set (image-level, so no annotation leakage between splits; every annotated image keeps all its boxes together).

Protocol: seeded RNG (42), 90/10 split, then a coverage pass guarantees every class has at least one validation image (moved atomically with all its annotations). Output `split.json` is attached by 03/04/05 via `kernel_sources`.

In [ ]:
import json, glob
from pathlib import Path
import numpy as np, pandas as pd

p = Path(glob.glob('/kaggle/input/**/train_dataset.json', recursive=True)[0])
data = json.load(open(p))
ids = [i['id'] for i in data['images']]
rng = np.random.default_rng(42)
rng.shuffle(ids)
n_val = max(int(round(0.10 * len(ids))), 32)
val_set = set(ids[:n_val])
train_ids = [i for i in ids if i not in val_set]

anns_by_img = {}
for a in data['annotations']:
    anns_by_img.setdefault(a['image_id'], []).append(a)
cats = {c['id']: c['name'] for c in data['categories']}
by_cls = {c: [i for i in ids if i in anns_by_img and any(a['category_id'] == c for a in anns_by_img[i])]
          for c in cats}

# coverage pass: rare classes first
moved = []
for c in sorted(by_cls, key=lambda k: len(by_cls[k])):
    if not any(i in val_set for i in by_cls[c]):
        cand = [i for i in by_cls[c] if i not in val_set]
        if cand:
            pick = cand[0]
            val_set.add(pick)
            moved.append(cats[c])
train_ids = [i for i in train_ids if i not in val_set]
val_ids = sorted(val_set)

json.dump({'seed': 42, 'val_fraction': 0.10, 'train_ids': train_ids, 'val_ids': val_ids},
          open('/kaggle/working/split.json', 'w'))

tr_s, va_s = set(train_ids), set(val_ids)
tr_a = [a for a in data['annotations'] if a['image_id'] in tr_s]
va_a = [a for a in data['annotations'] if a['image_id'] in va_s]
print(f'train: {len(train_ids)} imgs / {len(tr_a)} anns | val: {len(val_ids)} imgs / {len(va_a)} anns')
print('classes moved an image to val for coverage:', moved or 'none')

tt = pd.Series([a['category_id'] for a in tr_a]).value_counts()
vv = pd.Series([a['category_id'] for a in va_a]).value_counts()
tab = pd.DataFrame({'train': tt, 'val': vv}).fillna(0).astype(int)
tab.index = [cats[i] for i in tab.index]
tab = tab.sort_values('train', ascending=False)
tab.to_csv('/kaggle/working/table_split_class_counts.csv')
print(f'val instance share: {len(va_a)/len(tr_a)*100:.2f}%')
print(f'classes with 0 val instances: {int((tab.val == 0).sum())}')
tab